# Convolution + Attention Hybrid Experiments

Core ML Task 4 notebook. This is based on the baseline Transformer structure, with placeholders for the empirically best attention mechanism and best positional embedding system. Fill those placeholders after Task 2 and Task 3 results are known.

Implemented hybrid designs:

1. Causal depthwise-separable Conv1D before every attention block.
2. Alternating blocks where some attention layers are replaced by causal depthwise-separable Conv1D blocks.

# Setup

In [ ]:
!pip install -q datasets transformers tqdm pandas

# Imports

In [ ]:
import math
import os
import random
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Drive Logging

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = "/content/drive/MyDrive/SAiDL_Assignment/core_ml_conv_attention_hybrids"
except Exception:
    LOG_DIR = "conv_attention_hybrid_logs"

os.makedirs(LOG_DIR, exist_ok=True)
print("Logging to:", LOG_DIR)

# Config

In [ ]:
class Config:
    vocab_size = 50257
    block_size = 1024
    n_layer = 4
    n_head = 4
    n_embd = 256
    dropout = 0.1
    learning_rate = 3e-4

    epochs = 15
    seed = 42
    batch_size = 8

    window_size = 256

    # Convolution settings.
    conv_kernel_size = 5

    # For the alternating design, replace attention on these layer indices.
    # With 4 layers, this pattern keeps attention in layers 0 and 2, and uses conv in 1 and 3.
    conv_replacement_layers = [1, 3]


def make_config():
    return SimpleNamespace(
        vocab_size=Config.vocab_size,
        block_size=Config.block_size,
        n_layer=Config.n_layer,
        n_head=Config.n_head,
        n_embd=Config.n_embd,
        dropout=Config.dropout,
        learning_rate=Config.learning_rate,
        conv_kernel_size=Config.conv_kernel_size,
        conv_replacement_layers=Config.conv_replacement_layers,
        window_size=Config.window_size,
    )

# Load Dataset + Tokenizer

In [ ]:
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Tokenize + Chunk

In [ ]:
def tokenize(example):
    return tokenizer(example["text"])


tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])


def group_texts(examples):
    block_size = Config.block_size
    concatenated = sum(examples["input_ids"], [])
    total_length = (len(concatenated) // block_size) * block_size

    input_ids = [
        concatenated[i:i + block_size]
        for i in range(0, total_length, block_size)
    ]

    return {"input_ids": input_ids, "labels": input_ids.copy()}


lm_datasets = tokenized.map(
    group_texts,
    batched=True,
    remove_columns=tokenized["train"].column_names,
)

# DataLoader

In [ ]:
def collate(batch):
    input_ids = torch.tensor([x["input_ids"] for x in batch], dtype=torch.long)
    labels = torch.tensor([x["labels"] for x in batch], dtype=torch.long)
    return input_ids, labels


train_loader = torch.utils.data.DataLoader(
    lm_datasets["train"],
    batch_size=Config.batch_size,
    shuffle=True,
    collate_fn=collate,
)

val_loader = torch.utils.data.DataLoader(
    lm_datasets["validation"],
    batch_size=Config.batch_size,
    shuffle=False,
    collate_fn=collate,
)

# TODO: Best Positional Embedding System

In [ ]:
# Needed by BestPositionProvider
def get_alibi_slopes(n_heads):
    def get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(n_heads).is_integer():
        return torch.tensor(get_slopes_power_of_2(n_heads), dtype=torch.float32)

    closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))
    slopes = get_slopes_power_of_2(closest_power_of_2)
    extra = get_alibi_slopes(2 * closest_power_of_2)[0::2]
    slopes.extend(extra[: n_heads - closest_power_of_2].tolist())
    return torch.tensor(slopes, dtype=torch.float32)


class BestPositionProvider(nn.Module):
    """ALiBi positional encoding — empirically best from Task 3.

    Adds a fixed, per-head linear distance penalty to attention scores.
    No learnable parameters. Extrapolates to longer sequences without
    degradation because the bias is a closed-form function of distance,
    not a lookup table.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config

        slopes = get_alibi_slopes(config.n_head).view(1, config.n_head, 1, 1)
        positions = torch.arange(config.block_size)
        distance = (positions[:, None] - positions[None, :]).float()
        alibi_bias = -slopes * distance.view(1, 1, config.block_size, config.block_size)
        self.register_buffer("alibi_bias", alibi_bias)

    def add_to_embeddings(self, x):
        # ALiBi is applied at the attention score level, not the input level.
        return x

    def modify_qk(self, q, k):
        # ALiBi does not modify Q or K.
        return q, k

    def attention_bias(self, T, device, dtype):
        # Slice the pre-computed buffer to the current sequence length and
        # cast to match the attention score dtype (important under AMP).
        return self.alibi_bias[:, :, :T, :T].to(dtype=dtype)

# TODO: Best Attention Variant

In [ ]:
class BestAttention(nn.Module):
    # Sliding Window

    def __init__(self, config, position_provider):
        super().__init__()
        self.config = config
        self.position_provider = position_provider

        # Pasted:
        assert config.n_embd % config.n_head == 0

        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.window_size = config.window_size

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        positions = torch.arange(config.block_size)
        distance = positions[:, None] - positions[None, :]
        local_causal_mask = (distance >= 0) & (distance < self.window_size)

        self.register_buffer(
            "mask",
            local_causal_mask.unsqueeze(0).unsqueeze(0),
        )

    def forward(self, x):
        # Pasted:
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Needed for interfacing with ALiBi
        bias = self.position_provider.attention_bias(T, x.device, att.dtype)
        if bias is not None:
            att = att + bias

        att = att.masked_fill(~self.mask[:, :, :T, :T], float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

# Convolution Components

In [ ]:
class CausalDepthwiseSeparableConv1D(nn.Module):
    """Causal depthwise-separable Conv1D over the sequence dimension."""

    def __init__(self, config):
        super().__init__()
        self.kernel_size = config.conv_kernel_size
        self.depthwise = nn.Conv1d(
            in_channels=config.n_embd,
            out_channels=config.n_embd,
            kernel_size=self.kernel_size,
            groups=config.n_embd,
            bias=True,
        )
        self.pointwise = nn.Conv1d(
            in_channels=config.n_embd,
            out_channels=config.n_embd,
            kernel_size=1,
            bias=True,
        )
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # x: (B, T, C)
        x = x.transpose(1, 2)  # (B, C, T)
        x = F.pad(x, (self.kernel_size - 1, 0))
        x = self.depthwise(x)
        x = F.gelu(x)
        x = self.pointwise(x)
        x = x.transpose(1, 2)  # (B, T, C)
        return self.dropout(x)

# Feedforward

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

# Hybrid Block Design A: Conv Before Attention

In [ ]:
class ConvBeforeAttentionBlock(nn.Module):
    """Adds a causal Conv1D residual branch before each attention branch."""

    def __init__(self, config, layer_idx, position_provider):
        super().__init__()
        self.ln_conv = nn.LayerNorm(config.n_embd)
        self.conv = CausalDepthwiseSeparableConv1D(config)

        self.ln_attn = nn.LayerNorm(config.n_embd)
        self.attn = BestAttention(config, position_provider)

        self.ln_ff = nn.LayerNorm(config.n_embd)
        self.ff = FeedForward(config)

    def forward(self, x):
        x = x + self.conv(self.ln_conv(x))
        x = x + self.attn(self.ln_attn(x))
        x = x + self.ff(self.ln_ff(x))
        return x

# Hybrid Block Design B: Alternating Conv / Attention Layers

In [ ]:
class AlternatingConvAttentionBlock(nn.Module):
    """Replaces selected attention layers with causal Conv1D blocks."""

    def __init__(self, config, layer_idx, position_provider):
        super().__init__()
        self.use_conv = layer_idx in set(config.conv_replacement_layers)

        self.ln_main = nn.LayerNorm(config.n_embd)
        if self.use_conv:
            self.main = CausalDepthwiseSeparableConv1D(config)
        else:
            self.main = BestAttention(config, position_provider)

        self.ln_ff = nn.LayerNorm(config.n_embd)
        self.ff = FeedForward(config)

    def forward(self, x):
        x = x + self.main(self.ln_main(x))
        x = x + self.ff(self.ln_ff(x))
        return x

# Plain Block with Best Attention and Positional Embeddings

In [ ]:
class PlainBlock(nn.Module):
    """Standard Transformer block using BestAttention + BestPositionProvider,
    with no convolutional component. This serves as the direct reference for
    Task 4c: isolates the effect of adding convolution by holding the attention
    variant and positional encoding fixed."""

    def __init__(self, config, layer_idx, position_provider):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = BestAttention(config, position_provider)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ff = FeedForward(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# Full Model

In [ ]:
class HybridTransformerModel(nn.Module):
    def __init__(self, config, block_cls):
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_provider = BestPositionProvider(config)

        self.blocks = nn.ModuleList([
            block_cls(config, layer_idx, self.position_provider)
            for layer_idx in range(config.n_layer)
        ])

        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size

        x = self.token_emb(idx)
        x = self.position_provider.add_to_embeddings(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
                targets[:, 1:].contiguous().view(-1),
            )

        return logits, loss

# Training + Evaluation Helpers

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def get_peak_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0.0


def evaluate(model):
    model.eval()
    losses = []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            _, loss = model(x, y)
            losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)


def measure_inference_throughput(model, max_batches=20):
    model.eval()
    total_tokens = 0

    sync_cuda()
    start_time = time.perf_counter()

    with torch.no_grad():
        for batch_idx, (x, _) in enumerate(val_loader):
            if batch_idx >= max_batches:
                break
            x = x.to(device)
            model(x)
            total_tokens += x.numel()

    sync_cuda()
    elapsed = time.perf_counter() - start_time
    model.train()

    return total_tokens / max(elapsed, 1e-9)


def save_checkpoint(path, model, optimizer, epoch, metrics):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics": metrics,
        },
        path,
    )

# Experiment Runner

In [ ]:
def train_hybrid_experiment(
    design_name,
    block_cls,
    epochs=Config.epochs,
    seed=Config.seed,
):
    set_seed(seed)
    config = make_config()

    model = HybridTransformerModel(config, block_cls).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

    run_id = f"{design_name}_ctx{Config.block_size}_seed{seed}"
    metrics_path = os.path.join(LOG_DIR, f"{run_id}_metrics.csv")
    checkpoint_path = os.path.join(LOG_DIR, f"{run_id}_latest.pt")

    start_epoch = 0
    training_metrics = []

    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        if os.path.exists(metrics_path):
            training_metrics = pd.read_csv(metrics_path).to_dict("records")
        print(f"Resumed {design_name} from epoch {start_epoch}")


    for epoch in range(start_epoch, epochs):
        model.train()
        pbar = tqdm(train_loader, desc=f"{design_name} seed {seed} epoch {epoch}")

        total_loss = 0.0
        total_tokens = 0

        reset_peak_memory()
        sync_cuda()
        start_time = time.perf_counter()

        for x, y in pbar:
            x, y = x.to(device), y.to(device)

            _, loss = model(x, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_tokens = y[:, 1:].numel()
            total_tokens += batch_tokens
            total_loss += loss.item()

            pbar.set_postfix(loss=f"{loss.item():.4f}")

        sync_cuda()
        epoch_time = time.perf_counter() - start_time

        train_loss = total_loss / len(train_loader)
        train_throughput = total_tokens / epoch_time
        val_loss = evaluate(model)
        val_perplexity = math.exp(min(val_loss, 20.0))
        inference_tokens_per_sec = measure_inference_throughput(model)
        peak_gpu_memory_mb = get_peak_memory_mb()

        epoch_metrics = {
            "design_name": design_name,
            "seed": seed,
            "epoch": epoch,
            "context_length": Config.block_size,
            "batch_size": Config.batch_size,
            "conv_kernel_size": Config.conv_kernel_size,
            "conv_replacement_layers": str(Config.conv_replacement_layers),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_perplexity": val_perplexity,
            "epoch_time_sec": epoch_time,
            "train_throughput_tokens_per_sec": train_throughput,
            "inference_throughput_tokens_per_sec": inference_tokens_per_sec,
            "peak_gpu_memory_mb": peak_gpu_memory_mb,
        }

        training_metrics.append(epoch_metrics)

        pd.DataFrame(training_metrics).to_csv(metrics_path, index=False)
        save_checkpoint(checkpoint_path, model, optimizer, epoch, epoch_metrics)

        print(epoch_metrics)
        print("saved metrics:", metrics_path)
        print("saved checkpoint:", checkpoint_path)

    return pd.DataFrame(training_metrics)

# Run: Conv Before Every Attention Block

In [ ]:
# conv_before_attention_results = train_hybrid_experiment(
#     design_name="conv_before_attention",
#     block_cls=ConvBeforeAttentionBlock,
#     epochs=Config.epochs,
#     seed=Config.seed,
# )

# display(conv_before_attention_results)

# Run: Alternating Conv / Attention Layers

In [ ]:
# alternating_conv_attention_results = train_hybrid_experiment(
#     design_name="alternating_conv_attention",
#     block_cls=AlternatingConvAttentionBlock,
#     epochs=Config.epochs,
#     seed=Config.seed,
# )

# display(alternating_conv_attention_results)

# Run: Plain Results Combining Best Attention and PosEmb

In [ ]:
plain_results = train_hybrid_experiment(
    design_name="plain_best_attn_pe",
    block_cls=PlainBlock,
    epochs=Config.epochs,
    seed=Config.seed,
)

display(plain_results)

# Aggregate Saved Results

In [ ]:
metric_files = [
    os.path.join(LOG_DIR, name)
    for name in os.listdir(LOG_DIR)
    if name.endswith("_metrics.csv")
]

if metric_files:
    all_metrics = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
    display(all_metrics)

    final_epoch_metrics = all_metrics.sort_values("epoch").groupby(
        ["design_name", "seed"],
        as_index=False,
    ).tail(1)

    display(final_epoch_metrics.sort_values("val_perplexity"))

    aggregate_path = os.path.join(LOG_DIR, "conv_attention_hybrids_all_metrics.csv")
    all_metrics.to_csv(aggregate_path, index=False)
    print("saved aggregate metrics:", aggregate_path)
else:
    print("No metric files found yet in", LOG_DIR)